![](/Workspace/Users/sunnygupta2508@gmail.com/Databricks-Certified-Data-Engineer-Pro/Includes/images/gold.png)

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%sql
create view if not exists countries_stats_vw as (
  select country, date_trunc("DD", order_timestamp):: date as order_date, count(order_id) as orders_count, sum(quantity) as books_count
  from customers_orders
  group by all
)

In [0]:
%sql
select *
from countries_stats_vw

In [0]:
from pyspark.sql import functions as F

query = (
        spark.readStream
                .table("books_sales")
                .withWatermark("order_timestamp", "10 minutes")
                .groupBy(
                    F.window("order_timestamp", "5 minutes").alias("time"), "author"
                )
                .agg(
                    F.count("order_id").alias("orders_count"),
                    F.avg("quantity").alias("avg_quantity")
                )
            .writeStream
                .option("checkpointLocation", f"{bookstore.checkpoint_path}/authors_stat")
                .trigger(availableNow=True)
                .table("authors_stat")               
)
query.awaitTermination()

In [0]:
%sql
select *
from authors_stat